# 🌊 Frequency Domain Image Enhancement Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/[your-repo]/blob/main/04%20-%20Image%20Enhancement%20-%20Frequency%20Domain/notebooks/frequency_domain_workshop.ipynb)

**CMSC 178IP - Digital Image Processing**

## 🎯 Workshop Objectives

In this interactive workshop, you will:
- Master the 2D Fourier Transform and its interpretation
- Design and apply frequency domain filters
- Implement homomorphic filtering for illumination correction
- Solve real-world image enhancement problems
- Compare spatial vs. frequency domain approaches

**⏱️ Estimated Duration:** 45-60 minutes

## 📦 Setup & Imports

In [ ]:
# Install required packages if running in Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install opencv-python-headless scikit-image
except ImportError:
    IN_COLAB = False

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cv2
from skimage import data, filters, restoration
from scipy.fft import fft2, ifft2, fftshift, ifftshift
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ All packages loaded successfully!")
print(f"📍 Running in {'Google Colab' if IN_COLAB else 'Local Environment'}")

## Part 1: Understanding the 2D Fourier Transform 🔍

The 2D Discrete Fourier Transform is the foundation of frequency domain image processing. Let's explore its properties and interpretation.

In [ ]:
# Load a sample image
image = data.camera().astype(float) / 255.0
print(f"📊 Image shape: {image.shape}")

# Compute 2D FFT
f_transform = fft2(image)
f_shifted = fftshift(f_transform)  # Shift zero frequency to center

# Extract magnitude and phase
magnitude = np.abs(f_shifted)
phase = np.angle(f_shifted)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original image
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Magnitude spectrum (log scale for better visibility)
axes[0, 1].imshow(np.log(magnitude + 1), cmap='hot')
axes[0, 1].set_title('Magnitude Spectrum (Log Scale)')
axes[0, 1].axis('off')

# Phase spectrum
axes[1, 0].imshow(phase, cmap='gray')
axes[1, 0].set_title('Phase Spectrum')
axes[1, 0].axis('off')

# Reconstruction check
reconstructed = np.real(ifft2(ifftshift(f_shifted)))
axes[1, 1].imshow(reconstructed, cmap='gray')
axes[1, 1].set_title('Reconstructed Image')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Verify perfect reconstruction
reconstruction_error = np.mean(np.abs(image - reconstructed))
print(f"🔍 Reconstruction error: {reconstruction_error:.2e} (should be very small)")

### 🧠 Key Insights

- **Magnitude spectrum** shows frequency content intensity
- **Phase spectrum** contains spatial information
- **Center (DC component)** represents average image intensity
- **Radial distance from center** represents frequency magnitude
- **Perfect reconstruction** validates the transform

## Part 2: Frequency Domain Filtering 🔧

Now let's design and apply various frequency domain filters.

In [ ]:
def create_frequency_filters(shape, d0=0.3):
    """Create various frequency domain filters."""
    rows, cols = shape
    center_row, center_col = rows // 2, cols // 2
    
    # Create distance matrix
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    d_normalized = distance / (min(rows, cols) / 2)
    
    # Filter designs
    filters = {}
    
    # Low-pass filters
    filters['ideal_lp'] = (d_normalized <= d0).astype(float)
    filters['gaussian_lp'] = np.exp(-(d_normalized**2) / (2 * d0**2))
    filters['butterworth_lp'] = 1 / (1 + (d_normalized / d0)**(2*2))  # n=2
    
    # High-pass filters
    filters['ideal_hp'] = (d_normalized > d0).astype(float)
    filters['gaussian_hp'] = 1 - filters['gaussian_lp']
    filters['butterworth_hp'] = 1 / (1 + (d0 / (d_normalized + 1e-10))**(2*2))
    
    return filters, d_normalized

# Create filters
filters, d_norm = create_frequency_filters(image.shape)

# Visualize filter responses
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

filter_names = ['ideal_lp', 'gaussian_lp', 'butterworth_lp', 
                'ideal_hp', 'gaussian_hp', 'butterworth_hp']
filter_titles = ['Ideal Low-pass', 'Gaussian Low-pass', 'Butterworth Low-pass',
                'Ideal High-pass', 'Gaussian High-pass', 'Butterworth High-pass']

for i, (name, title) in enumerate(zip(filter_names, filter_titles)):
    row, col = i // 3, i % 3
    axes[row, col].imshow(filters[name], cmap='gray')
    axes[row, col].set_title(title)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("🎛️ Filter bank created successfully!")

In [ ]:
# Apply filters and compare results
def apply_frequency_filter(image, filter_response):
    """Apply a frequency domain filter to an image."""
    # Transform to frequency domain
    F = fftshift(fft2(image))
    
    # Apply filter
    F_filtered = F * filter_response
    
    # Transform back to spatial domain
    filtered_image = np.real(ifft2(ifftshift(F_filtered)))
    
    return filtered_image

# Test different filters
test_filters = ['gaussian_lp', 'gaussian_hp', 'butterworth_lp']
test_titles = ['Gaussian Low-pass\n(Smoothing)', 'Gaussian High-pass\n(Edge Enhancement)', 'Butterworth Low-pass\n(Smooth Cutoff)']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Apply filters
positions = [(0, 1), (1, 0), (1, 1)]
for i, (filter_name, title) in enumerate(zip(test_filters, test_titles)):
    filtered = apply_frequency_filter(image, filters[filter_name])
    row, col = positions[i]
    axes[row, col].imshow(filtered, cmap='gray')
    axes[row, col].set_title(title)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("🔍 Filter effects demonstrated!")

## Part 3: Homomorphic Filtering for Illumination Correction 💡

Homomorphic filtering separates illumination and reflectance components using the logarithmic transform.

In [ ]:
def homomorphic_filter(image, gamma_l=0.3, gamma_h=2.0, d0=0.25, c=1.0):
    """
    Apply homomorphic filtering to correct illumination.
    
    Parameters:
    - gamma_l: Low frequency gain (illumination suppression)
    - gamma_h: High frequency gain (reflectance enhancement) 
    - d0: Cutoff frequency
    - c: Filter sharpness
    """
    # Step 1: Take natural logarithm
    log_image = np.log(image + 1e-10)  # Add small constant to avoid log(0)
    
    # Step 2: Apply 2D FFT
    F = fftshift(fft2(log_image))
    
    # Step 3: Design homomorphic filter
    rows, cols = image.shape
    center_row, center_col = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    d_normalized = distance / (min(rows, cols) / 2)
    
    # Homomorphic filter function
    H = (gamma_h - gamma_l) * (1 - np.exp(-c * (d_normalized / d0)**2)) + gamma_l
    
    # Step 4: Apply filter
    F_filtered = F * H
    
    # Step 5: Inverse FFT
    log_filtered = np.real(ifft2(ifftshift(F_filtered)))
    
    # Step 6: Exponential to get final result
    result = np.exp(log_filtered)
    
    return result, H, log_image

# Create a test image with uneven illumination
test_image = data.camera().astype(float) / 255.0

# Simulate uneven illumination
y, x = np.ogrid[:test_image.shape[0], :test_image.shape[1]]
center_y, center_x = test_image.shape[0] // 2, test_image.shape[1] // 2

# Gaussian illumination fall-off
illumination = np.exp(-((x - center_x)**2 + (y - center_y)**2) / (2 * (min(test_image.shape) // 3)**2))
illumination = 0.2 + 0.8 * illumination  # Scale to reasonable range

# Apply illumination
illuminated_image = test_image * illumination

# Apply homomorphic filtering
corrected, homo_filter, log_img = homomorphic_filter(illuminated_image)

# Normalize for display
corrected = (corrected - corrected.min()) / (corrected.max() - corrected.min())

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Top row: Image processing pipeline
axes[0, 0].imshow(test_image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(illuminated_image, cmap='gray')
axes[0, 1].set_title('Unevenly Illuminated')
axes[0, 1].axis('off')

axes[0, 2].imshow(corrected, cmap='gray')
axes[0, 2].set_title('Homomorphic Corrected')
axes[0, 2].axis('off')

# Bottom row: Analysis
axes[1, 0].imshow(illumination, cmap='gray')
axes[1, 0].set_title('Illumination Pattern')
axes[1, 0].axis('off')

axes[1, 1].imshow(homo_filter, cmap='gray')
axes[1, 1].set_title('Homomorphic Filter')
axes[1, 1].axis('off')

# Profile comparison
center_row = test_image.shape[0] // 2
axes[1, 2].plot(test_image[center_row, :], 'b-', linewidth=2, label='Original')
axes[1, 2].plot(illuminated_image[center_row, :], 'r-', linewidth=1, label='Illuminated')
axes[1, 2].plot(corrected[center_row, :], 'g-', linewidth=2, label='Corrected')
axes[1, 2].set_title('Intensity Profile Comparison')
axes[1, 2].set_xlabel('Pixel Position')
axes[1, 2].set_ylabel('Intensity')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantitative assessment
contrast_before = np.std(illuminated_image)
contrast_after = np.std(corrected)
improvement = contrast_after / contrast_before

print(f"📊 Contrast improvement: {improvement:.2f}x")
print("✅ Homomorphic filtering successfully corrected illumination!")

## Part 4: Noise Removal with Notch Filtering 🎯

Frequency domain excels at removing periodic noise that appears as distinct peaks in the spectrum.

In [ ]:
def add_periodic_noise(image, frequencies=[(40, 0), (0, 30)], amplitudes=[0.3, 0.2]):
    """Add periodic noise to an image."""
    y, x = np.ogrid[:image.shape[0], :image.shape[1]]
    noise = np.zeros_like(image)
    
    for (fx, fy), amp in zip(frequencies, amplitudes):
        if fx > 0:
            noise += amp * np.sin(2 * np.pi * fx * x / image.shape[1])
        if fy > 0:
            noise += amp * np.sin(2 * np.pi * fy * y / image.shape[0])
    
    return image + noise

def create_notch_filter(shape, notch_locations, notch_radius=8):
    """Create a notch filter to remove specific frequencies."""
    rows, cols = shape
    center_row, center_col = rows // 2, cols // 2
    
    # Start with all-pass filter
    notch_filter = np.ones((rows, cols), dtype=float)
    
    # Create coordinate matrices
    y, x = np.ogrid[:rows, :cols]
    
    # Add notches at specified locations
    for (offset_y, offset_x) in notch_locations:
        # Positive frequency location
        notch_y = center_row + offset_y
        notch_x = center_col + offset_x
        
        if 0 <= notch_y < rows and 0 <= notch_x < cols:
            mask = (x - notch_x)**2 + (y - notch_y)**2 <= notch_radius**2
            notch_filter[mask] = 0
        
        # Negative frequency location (symmetric)
        notch_y = center_row - offset_y
        notch_x = center_col - offset_x
        
        if 0 <= notch_y < rows and 0 <= notch_x < cols:
            mask = (x - notch_x)**2 + (y - notch_y)**2 <= notch_radius**2
            notch_filter[mask] = 0
    
    # Apply Gaussian smoothing to reduce ringing
    notch_filter = gaussian_filter(notch_filter, sigma=1)
    
    return notch_filter

# Create noisy image
clean_image = data.camera().astype(float) / 255.0
noisy_image = add_periodic_noise(clean_image)
noisy_image = np.clip(noisy_image, 0, 1)

# Analyze frequency content
F_clean = fftshift(fft2(clean_image))
F_noisy = fftshift(fft2(noisy_image))

# Design notch filter based on known noise frequencies
notch_locations = [(0, 40), (30, 0)]  # Corresponding to our added noise
notch_filter = create_notch_filter(clean_image.shape, notch_locations)

# Apply notch filter
F_filtered = F_noisy * notch_filter
filtered_image = np.real(ifft2(ifftshift(F_filtered)))
filtered_image = np.clip(filtered_image, 0, 1)

# Visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 10))

# Top row: Images
axes[0, 0].imshow(clean_image, cmap='gray')
axes[0, 0].set_title('Original Clean Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_image, cmap='gray')
axes[0, 1].set_title('Noisy Image')
axes[0, 1].axis('off')

axes[0, 2].imshow(notch_filter, cmap='gray')
axes[0, 2].set_title('Notch Filter')
axes[0, 2].axis('off')

axes[0, 3].imshow(filtered_image, cmap='gray')
axes[0, 3].set_title('Filtered Image')
axes[0, 3].axis('off')

# Bottom row: Frequency spectra
axes[1, 0].imshow(np.log(np.abs(F_clean) + 1), cmap='hot')
axes[1, 0].set_title('Clean Spectrum')
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log(np.abs(F_noisy) + 1), cmap='hot')
axes[1, 1].set_title('Noisy Spectrum')
axes[1, 1].axis('off')

# Mark noise peaks
center_row, center_col = clean_image.shape[0] // 2, clean_image.shape[1] // 2
for (offset_y, offset_x) in notch_locations:
    axes[1, 1].plot(center_col + offset_x, center_row + offset_y, 'r*', markersize=10)
    axes[1, 1].plot(center_col - offset_x, center_row - offset_y, 'r*', markersize=10)

axes[1, 2].imshow(np.log(np.abs(F_filtered) + 1), cmap='hot')
axes[1, 2].set_title('Filtered Spectrum')
axes[1, 2].axis('off')

# Error analysis
error_noisy = np.abs(clean_image - noisy_image)
error_filtered = np.abs(clean_image - filtered_image)

axes[1, 3].plot(error_noisy.mean(axis=0), 'r-', linewidth=2, label='Noisy Error')
axes[1, 3].plot(error_filtered.mean(axis=0), 'g-', linewidth=2, label='Filtered Error')
axes[1, 3].set_title('Error Profiles')
axes[1, 3].set_xlabel('Column')
axes[1, 3].set_ylabel('Mean Absolute Error')
axes[1, 3].legend()
axes[1, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quality metrics
mse_noisy = np.mean((clean_image - noisy_image)**2)
mse_filtered = np.mean((clean_image - filtered_image)**2)
psnr_noisy = 20 * np.log10(1.0 / np.sqrt(mse_noisy))
psnr_filtered = 20 * np.log10(1.0 / np.sqrt(mse_filtered))

print(f"📊 PSNR improvement: {psnr_noisy:.1f} dB → {psnr_filtered:.1f} dB")
print(f"📈 Improvement: {psnr_filtered - psnr_noisy:.1f} dB")
print("✅ Periodic noise successfully removed!")

## Part 5: Advanced Techniques - High-Frequency Emphasis 🚀

High-frequency emphasis enhances details while preserving overall image characteristics.

In [ ]:
def high_frequency_emphasis(image, gamma_l=0.5, gamma_h=2.0, d0=0.3):
    """Apply high-frequency emphasis filtering."""
    # Transform to frequency domain
    F = fftshift(fft2(image))
    
    # Create distance matrix
    rows, cols = image.shape
    center_row, center_col = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    d_normalized = distance / (min(rows, cols) / 2)
    
    # High-frequency emphasis filter
    # H(u,v) = γ_l + (γ_h - γ_l) * [1 - exp(-(D(u,v)/D0)^2)]
    high_pass = 1 - np.exp(-(d_normalized / d0)**2)
    hfe_filter = gamma_l + (gamma_h - gamma_l) * high_pass
    
    # Apply filter
    F_enhanced = F * hfe_filter
    
    # Transform back
    enhanced = np.real(ifft2(ifftshift(F_enhanced)))
    
    return enhanced, hfe_filter

# Create a slightly blurred image for enhancement
original = data.camera().astype(float) / 255.0
blurred = gaussian_filter(original, sigma=1.5)

# Apply different enhancement levels
mild_enhanced, mild_filter = high_frequency_emphasis(blurred, gamma_l=0.7, gamma_h=1.5)
moderate_enhanced, mod_filter = high_frequency_emphasis(blurred, gamma_l=0.5, gamma_h=2.0)
strong_enhanced, strong_filter = high_frequency_emphasis(blurred, gamma_l=0.3, gamma_h=2.5)

# Normalize results
results = [mild_enhanced, moderate_enhanced, strong_enhanced]
for i, result in enumerate(results):
    results[i] = np.clip(result, 0, 1)

# Visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 10))

# Top row: Enhancement results
axes[0, 0].imshow(original, cmap='gray')
axes[0, 0].set_title('Original Sharp')
axes[0, 0].axis('off')

axes[0, 1].imshow(blurred, cmap='gray')
axes[0, 1].set_title('Blurred Input')
axes[0, 1].axis('off')

axes[0, 2].imshow(results[1], cmap='gray')  # Moderate enhancement
axes[0, 2].set_title('HFE Enhanced')
axes[0, 2].axis('off')

axes[0, 3].imshow(results[2], cmap='gray')  # Strong enhancement
axes[0, 3].set_title('Strong Enhancement')
axes[0, 3].axis('off')

# Bottom row: Filter analysis
filters_to_show = [mild_filter, mod_filter, strong_filter]
filter_labels = ['Mild (γl=0.7, γh=1.5)', 'Moderate (γl=0.5, γh=2.0)', 'Strong (γl=0.3, γh=2.5)']

for i, (filt, label) in enumerate(zip(filters_to_show, filter_labels)):
    axes[1, i].imshow(filt, cmap='gray')
    axes[1, i].set_title(label)
    axes[1, i].axis('off')

# Profile comparison
center_row = original.shape[0] // 2
axes[1, 3].plot(original[center_row, :], 'b-', linewidth=2, label='Original')
axes[1, 3].plot(blurred[center_row, :], 'r-', linewidth=1, label='Blurred')
axes[1, 3].plot(results[1][center_row, :], 'g-', linewidth=2, label='HFE Enhanced')
axes[1, 3].set_title('Intensity Profile')
axes[1, 3].set_xlabel('Pixel Position')
axes[1, 3].set_ylabel('Intensity')
axes[1, 3].legend()
axes[1, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate sharpness metrics
def calculate_sharpness(img):
    """Calculate image sharpness using Laplacian variance."""
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    return np.var(laplacian)

sharpness_original = calculate_sharpness(original)
sharpness_blurred = calculate_sharpness(blurred)
sharpness_enhanced = calculate_sharpness(results[1])

print(f"📊 Sharpness Analysis:")
print(f"   Original: {sharpness_original:.2f}")
print(f"   Blurred: {sharpness_blurred:.2f}")
print(f"   Enhanced: {sharpness_enhanced:.2f}")
print(f"📈 Improvement: {sharpness_enhanced/sharpness_blurred:.2f}x")
print("✅ High-frequency emphasis successfully applied!")

## 🎓 Student Activity: Design Your Own Filter (15 minutes)

Now it's your turn! Design and test a custom frequency domain filter.

### Your Task:
1. **Choose a problem**: Select one of the following:
   - Remove specific noise pattern
   - Enhance edges in a particular direction
   - Create artistic effect (e.g., radial blur)
   - Combine multiple filtering techniques

2. **Design your filter**: Write code to create your custom filter
3. **Test and evaluate**: Apply to test images and assess results
4. **Optimize**: Adjust parameters for best performance

In [ ]:
# 🎯 YOUR TURN: Design a custom frequency domain filter

def create_custom_filter(shape, **params):
    """
    Design your custom frequency domain filter here!
    
    Example ideas:
    - Directional filter (enhance edges in specific direction)
    - Multi-band filter (different gains for different frequency ranges)
    - Adaptive filter (varies with local image properties)
    - Artistic filter (creates special visual effects)
    """
    rows, cols = shape
    center_row, center_col = rows // 2, cols // 2
    
    # Create coordinate matrices
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    angle = np.arctan2(y - center_row, x - center_col)
    
    # TODO: Design your filter here!
    # Example: Directional high-pass filter
    d_normalized = distance / (min(rows, cols) / 2)
    
    # Simple example - you can make this much more creative!
    custom_filter = np.ones_like(d_normalized)
    
    # Add your filter design here
    # ...
    
    return custom_filter

# Test your filter
test_image = data.camera().astype(float) / 255.0

# Create and apply your custom filter
my_filter = create_custom_filter(test_image.shape)
my_result = apply_frequency_filter(test_image, my_filter)

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(test_image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(my_filter, cmap='gray')
axes[1].set_title('Your Custom Filter')
axes[1].axis('off')

axes[2].imshow(my_result, cmap='gray')
axes[2].set_title('Filter Result')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("🎨 Test your custom filter design!")
print("💡 Hint: Try modifying the filter based on distance, angle, or both!")

<details>
<summary><b>🔍 Click here for solution examples</b></summary>

```python
# Solution Example 1: Directional Edge Enhancement
def directional_filter(shape, direction=0, width=np.pi/4):
    rows, cols = shape
    center_row, center_col = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    angle = np.arctan2(y - center_row, x - center_col)
    
    # Create directional mask
    angle_diff = np.abs(angle - direction)
    angle_diff = np.minimum(angle_diff, 2*np.pi - angle_diff)
    directional_mask = (angle_diff <= width).astype(float)
    
    # Combine with high-pass
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    d_normalized = distance / (min(rows, cols) / 2)
    high_pass = 1 - np.exp(-(d_normalized / 0.3)**2)
    
    return 0.5 + 1.5 * high_pass * directional_mask

# Solution Example 2: Multi-band Filter
def multiband_filter(shape):
    rows, cols = shape
    center_row, center_col = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    d_normalized = distance / (min(rows, cols) / 2)
    
    # Different gains for different frequency bands
    low_band = d_normalized <= 0.2
    mid_band = (d_normalized > 0.2) & (d_normalized <= 0.6)
    high_band = d_normalized > 0.6
    
    filter_response = np.zeros_like(d_normalized)
    filter_response[low_band] = 0.8   # Suppress low frequencies
    filter_response[mid_band] = 1.2   # Enhance mid frequencies
    filter_response[high_band] = 0.5  # Moderate high frequencies
    
    return filter_response
```
</details>

## Part 6: Comparing Spatial vs. Frequency Domain 🔄

Let's compare the efficiency and effectiveness of spatial vs. frequency domain approaches.

In [ ]:
import time

def spatial_gaussian_filter(image, sigma=2):
    """Apply Gaussian filtering in spatial domain."""
    return gaussian_filter(image, sigma=sigma)

def frequency_gaussian_filter(image, sigma=2):
    """Apply Gaussian filtering in frequency domain."""
    # Create Gaussian filter in frequency domain
    rows, cols = image.shape
    center_row, center_col = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    distance = np.sqrt((x - center_col)**2 + (y - center_row)**2)
    
    # Convert spatial sigma to frequency domain
    d0 = 1.0 / (2 * np.pi * sigma) * min(rows, cols) / 2
    gaussian_filter_freq = np.exp(-(distance / d0)**2)
    
    return apply_frequency_filter(image, gaussian_filter_freq)

# Test on different image sizes
test_sizes = [128, 256, 512]
spatial_times = []
frequency_times = []

print("⏱️ Performance Comparison: Spatial vs. Frequency Domain")
print("=" * 60)

for size in test_sizes:
    # Create test image
    test_img = cv2.resize(data.camera().astype(float) / 255.0, (size, size))
    
    # Time spatial domain filtering
    start_time = time.time()
    for _ in range(5):  # Average over multiple runs
        spatial_result = spatial_gaussian_filter(test_img)
    spatial_time = (time.time() - start_time) / 5
    spatial_times.append(spatial_time)
    
    # Time frequency domain filtering
    start_time = time.time()
    for _ in range(5):
        frequency_result = frequency_gaussian_filter(test_img)
    frequency_time = (time.time() - start_time) / 5
    frequency_times.append(frequency_time)
    
    print(f"Size {size}x{size}: Spatial={spatial_time:.4f}s, Frequency={frequency_time:.4f}s")

# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Performance plot
axes[0].plot(test_sizes, spatial_times, 'bo-', linewidth=2, markersize=8, label='Spatial Domain')
axes[0].plot(test_sizes, frequency_times, 'ro-', linewidth=2, markersize=8, label='Frequency Domain')
axes[0].set_xlabel('Image Size')
axes[0].set_ylabel('Processing Time (seconds)')
axes[0].set_title('Performance Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Results comparison
test_img = cv2.resize(data.camera().astype(float) / 255.0, (256, 256))
spatial_result = spatial_gaussian_filter(test_img)
frequency_result = frequency_gaussian_filter(test_img)

axes[1].imshow(spatial_result, cmap='gray')
axes[1].set_title('Spatial Domain Result')
axes[1].axis('off')

axes[2].imshow(frequency_result, cmap='gray')
axes[2].set_title('Frequency Domain Result')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Quality comparison
difference = np.abs(spatial_result - frequency_result)
max_diff = np.max(difference)
mean_diff = np.mean(difference)

print(f"\n📊 Quality Comparison:")
print(f"   Maximum difference: {max_diff:.6f}")
print(f"   Mean difference: {mean_diff:.6f}")
print("\n📝 Key Insights:")
print("   • Frequency domain excels at selective filtering")
print("   • Spatial domain better for local operations")
print("   • FFT makes frequency operations feasible")
print("   • Choice depends on filter type and image size")

## 📚 Summary & Key Takeaways

Congratulations! You've successfully completed the Frequency Domain Image Enhancement workshop. Let's summarize what you've learned:

### 🎯 Core Concepts Mastered:
1. **2D Fourier Transform**: Understanding magnitude, phase, and frequency interpretation
2. **Filter Design**: Low-pass, high-pass, band-pass, and notch filters
3. **Homomorphic Filtering**: Separating illumination and reflectance
4. **Noise Removal**: Using notch filters for periodic interference
5. **Enhancement Techniques**: High-frequency emphasis and sharpening

### 🔧 Practical Skills Developed:
- ✅ Designing custom frequency domain filters
- ✅ Analyzing frequency content of images
- ✅ Solving real-world enhancement problems
- ✅ Evaluating filter performance quantitatively
- ✅ Choosing appropriate enhancement techniques

### 🚀 Next Steps:
- Experiment with different filter parameters
- Apply techniques to your own images
- Explore advanced topics like wavelets
- Study computational efficiency optimizations

### 📖 Additional Learning Resources:
- **Textbook**: Gonzalez & Woods "Digital Image Processing" Chapter 4
- **Online**: Interactive FFT visualizations
- **Software**: MATLAB Image Processing Toolbox, OpenCV
- **Research**: Recent papers on frequency domain techniques

---

**🎉 Well done! You're now equipped to tackle complex image enhancement challenges using frequency domain techniques!**

*Continue exploring and happy image processing! 📸✨*